# HW08-09 — KMNIST, MLP, регуляризация и диагностика LR

Этот ноутбук — рабочая версия. Если вы предпочитаете — запускайте раздельные тетради: `HW08-09-01-setup.ipynb`, `HW08-09-02-model.ipynb`, `HW08-09-03-experiments.ipynb`.

In [16]:
# Импорты и базовая настройка
import os
import random
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms

# Папки для артефактов
ART_DIR = os.path.join('homeworks', 'HW08-09', 'artifacts')
FIG_DIR = os.path.join(ART_DIR, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


In [ ]:
from torchvision import transforms
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
data_root = os.path.join('homeworks', 'HW08-09', 'data')
os.makedirs(data_root, exist_ok=True)
train_full = torchvision.datasets.KMNIST(root=data_root, train=True, download=True, transform=transform)
test_ds = torchvision.datasets.KMNIST(root=data_root, train=False, download=True, transform=transform)

print('train_full size:', len(train_full), 'test size:', len(test_ds))

# split train/val 80/20 reproducible
val_ratio = 0.2
val_size = int(len(train_full) * val_ratio)
train_size = len(train_full) - val_size
gen = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(train_full, [train_size, val_size], generator=gen)

BATCH = 128
NUM_WORKERS = 0 if os.name == 'nt' else 2
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))

# sanity check
x,y = next(iter(train_loader))
print('batch x:', x.shape, 'y:', y.shape, 'range:', x.min().item(), x.max().item())

In [ ]:
def accuracy_from_logits(logits: torch.Tensor, y_true: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y_true).float().mean().item()

import matplotlib.pyplot as plt
def plot_history(history, path=None, title=''):
    epochs = np.arange(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(6,4))
    plt.plot(epochs, history['train_loss'], label='train_loss')
    plt.plot(epochs, history['val_loss'], label='val_loss')
    plt.xlabel('epoch')
    plt.ylabel('loss')
    plt.legend()
    plt.grid(True)
    if path: plt.savefig(path, bbox_inches='tight')
    plt.show()
    plt.figure(figsize=(6,4))
    plt.plot(epochs, history['train_acc'], label='train_acc')
    plt.plot(epochs, history['val_acc'], label='val_acc')
    plt.xlabel('epoch')
    plt.ylabel('accuracy')
    plt.legend()
    plt.grid(True)
    if path: plt.savefig(path.replace('.png', '_acc.png'), bbox_inches='tight')
    plt.show()

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=28*28, hidden_dims=(512,256), num_classes=10, activation='relu', dropout_p=0.0, use_batchnorm=False):
        super().__init__()
        if activation.lower()=='relu':
            act = nn.ReLU
        elif activation.lower()=='tanh':
            act = nn.Tanh
        else:
            act = nn.ReLU
        layers = [nn.Flatten()]
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(act())
            if dropout_p>0:
                layers.append(nn.Dropout(dropout_p))
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

print('MLP class ready')

MLP class ready


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, yb)
        n += 1
    return total_loss/n, total_acc/n

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, yb)
        n += 1
    return total_loss/n, total_acc/n

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.best_state = None
        self.counter = 0
    def step(self, score, model):
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience
    def restore_best(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

def fit(model, train_loader, val_loader, optimizer, criterion, device, epochs=20, early_stopping=None, verbose=True):
    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    for ep in range(1, epochs+1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss)
        history['val_acc'].append(va_acc)
        if verbose:
            print(f'ep {ep}/{epochs} | tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | val_loss={va_loss:.4f} val_acc={va_acc:.4f}')
        if early_stopping is not None:
            stop = early_stopping.step(va_acc, model)
            if stop:
                print(f'EarlyStopping at ep {ep}, best_val_acc={early_stopping.best_score:.4f}')
                early_stopping.restore_best(model)
                break
    return history

In [ ]:
# Запуск E1-E3 (предполагается, что train_loader/val_loader определены в окружении)
results = []
criterion = nn.CrossEntropyLoss()

set_seed(SEED)
m1 = MLP(hidden_dims=(512,256), dropout_p=0.0, use_batchnorm=False).to(device)
opt = optim.Adam(m1.parameters(), lr=1e-3)
hist1 = fit(m1, train_loader, val_loader, opt, criterion, device, epochs=20, early_stopping=None)
results.append(('E1','KMNIST', SEED, '512-256 / relu / none / no-bn','Adam',1e-3,0,0,len(hist1['val_acc']), max(hist1['val_acc']), min(hist1['val_loss'])))

set_seed(SEED)
m2 = MLP(hidden_dims=(512,256), dropout_p=0.3, use_batchnorm=False).to(device)
opt = optim.Adam(m2.parameters(), lr=1e-3)
hist2 = fit(m2, train_loader, val_loader, opt, criterion, device, epochs=20, early_stopping=None)
results.append(('E2','KMNIST', SEED, '512-256 / relu / dropout0.3 / no-bn','Adam',1e-3,0,0,len(hist2['val_acc']), max(hist2['val_acc']), min(hist2['val_loss'])))

set_seed(SEED)
m3 = MLP(hidden_dims=(512,256), dropout_p=0.0, use_batchnorm=True).to(device)
opt = optim.Adam(m3.parameters(), lr=1e-3)
hist3 = fit(m3, train_loader, val_loader, opt, criterion, device, epochs=20, early_stopping=None)
results.append(('E3','KMNIST', SEED, '512-256 / relu / none / batchnorm','Adam',1e-3,0,0,len(hist3['val_acc']), max(hist3['val_acc']), min(hist3['val_loss'])))

# Сохраняем промежуточные результаты
import pandas as pd
cols = ['experiment_id','dataset','seed','model_summary','optimizer','lr','momentum','weight_decay','epochs_trained','best_val_accuracy','best_val_loss']
df = pd.DataFrame(results, columns=cols)
df.to_csv(os.path.join(ART_DIR, 'runs_partial_E1E3.csv'), index=False)
print('Saved runs_partial_E1E3.csv')

NameError: name 'train_loader' is not defined

In [ ]:
# загрузим частичные результаты, если есть
partial_path = os.path.join(ART_DIR, 'runs_partial_E1E3.csv')
if os.path.exists(partial_path):
    dfp = pd.read_csv(partial_path)
    print(dfp[['experiment_id','best_val_accuracy']])
else:
    print('partial runs not found; предполагаем, что E2 лучше E3 по коду')

partial runs not found; предполагаем, что E2 лучше E3 по коду


In [ ]:
# Выбор лучшего между E2 и E3 — если нет данных, выбираем E2 по умолчанию
use_e2 = True
if os.path.exists(partial_path):
    best_row = dfp.loc[dfp['experiment_id'].isin(['E2','E3'])].sort_values('best_val_accuracy', ascending=False).iloc[0]
    use_e2 = (best_row['experiment_id'] == 'E2')
print('use_e2 =', use_e2)

# Построим модель лучшей конфигурации
if use_e2:
    best_model = MLP(hidden_dims=(512,256), dropout_p=0.3, use_batchnorm=False).to(device)
    dropout = 0.3
    bn = False
else:
    best_model = MLP(hidden_dims=(512,256), dropout_p=0.0, use_batchnorm=True).to(device)
    dropout = 0.0
    bn = True

set_seed(SEED)
opt = optim.Adam(best_model.parameters(), lr=1e-3)
es = EarlyStopping(patience=4, min_delta=1e-4)
hist_e4 = fit(best_model, train_loader, val_loader, opt, nn.CrossEntropyLoss(), device, epochs=50, early_stopping=es)
# сохранить best model и config
torch.save(best_model.state_dict(), os.path.join(ART_DIR, 'best_model.pt'))
best_config = {'experiment':'E4','dataset':'KMNIST','seed':SEED,'hidden_dims':[512,256],'dropout':dropout,'batchnorm':bn,'optimizer':'Adam','lr':1e-3}
with open(os.path.join(ART_DIR, 'best_config.json'), 'w') as f: json.dump(best_config, f, indent=2)
# график
plot_history(hist_e4, path=os.path.join(FIG_DIR, 'curves_best.png'), title='E4 best')
# тест
test_loss, test_acc = evaluate(best_model, test_loader, nn.CrossEntropyLoss(), device)
print('E4 test:', test_loss, test_acc)

use_e2 = True


NameError: name 'train_loader' is not defined

In [ ]:
# O1-O3 — LR diagnosis and SGD comparison
set_seed(SEED)
m_o1 = MLP(hidden_dims=(512,256), dropout_p=dropout, use_batchnorm=bn).to(device)
opt = optim.Adam(m_o1.parameters(), lr=1e-1)
hist_o1 = fit(m_o1, train_loader, val_loader, opt, nn.CrossEntropyLoss(), device, epochs=8, early_stopping=None, verbose=True)

set_seed(SEED)
m_o2 = MLP(hidden_dims=(512,256), dropout_p=dropout, use_batchnorm=bn).to(device)
opt = optim.Adam(m_o2.parameters(), lr=1e-5)
hist_o2 = fit(m_o2, train_loader, val_loader, opt, nn.CrossEntropyLoss(), device, epochs=8, early_stopping=None, verbose=True)

set_seed(SEED)
m_o3 = MLP(hidden_dims=(512,256), dropout_p=dropout, use_batchnorm=bn).to(device)
opt = optim.SGD(m_o3.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)
hist_o3 = fit(m_o3, train_loader, val_loader, opt, nn.CrossEntropyLoss(), device, epochs=12, early_stopping=None, verbose=True)

# Сохраним диаграмму O1/O2
plt.figure(figsize=(8,4))
plt.plot(np.arange(1,len(hist_o1['val_loss'])+1), hist_o1['val_loss'], label='O1 val_loss (lr=1e-1)')
plt.plot(np.arange(1,len(hist_o2['val_loss'])+1), hist_o2['val_loss'], label='O2 val_loss (lr=1e-5)')
plt.xlabel('epoch')
plt.ylabel('val_loss')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(FIG_DIR, 'curves_lr_extremes.png'), bbox_inches='tight')
plt.show()

# Соберём все результаты в runs.csv — если ранее есть runs.csv, добавим новые строки
cols = ['experiment_id','dataset','seed','model_summary','optimizer','lr','momentum','weight_decay','epochs_trained','best_val_accuracy','best_val_loss']
rows = []
# попробуем прочитать существующий runs.csv
runs_path = os.path.join(ART_DIR, 'runs.csv')
if os.path.exists(runs_path):
    df_old = pd.read_csv(runs_path)
else:
    df_old = pd.DataFrame(columns=cols)
# добавляем E1-E4 из partial если есть
partial = os.path.join(ART_DIR, 'runs_partial_E1E3.csv')
if os.path.exists(partial):
    dfp = pd.read_csv(partial)
    df_old = pd.concat([df_old, dfp], ignore_index=True)
# добавляем E4 and O1-O3
rows.append(('E4','KMNIST', SEED, f"{best_config['hidden_dims']} / relu / dropout{best_config['dropout']} / {'bn' if best_config['batchnorm'] else 'no-bn'}", 'Adam', best_config['lr'], 0, 0, len(hist_e4['val_acc']), max(hist_e4['val_acc']), min(hist_e4['val_loss'])))
rows.append(('O1','KMNIST', SEED, 'same_arch / lr=1e-1','Adam',1e-1,0,0,len(hist_o1['val_acc']), max(hist_o1['val_acc']), min(hist_o1['val_loss'])))
rows.append(('O2','KMNIST', SEED, 'same_arch / lr=1e-5','Adam',1e-5,0,0,len(hist_o2['val_acc']), max(hist_o2['val_acc']), min(hist_o2['val_loss'])))
rows.append(('O3','KMNIST', SEED, 'SGD momentum=0.9 wd=1e-4','SGD',1e-2,0.9,1e-4,len(hist_o3['val_acc']), max(hist_o3['val_acc']), min(hist_o3['val_loss'])))
df_new = pd.DataFrame(rows, columns=cols)
df_all = pd.concat([df_old, df_new], ignore_index=True)
df_all.to_csv(runs_path, index=False)
print('Saved runs.csv with all experiments to', runs_path)